# Behavioral refusal screen
Run all cells on a Colab L4 or A100 with the HF_TOKEN secret enabled. Each completed record is saved to Drive. Reconnect and rerun to resume.

In [ ]:
from pathlib import Path
import json, os, sys
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_INPUT_FOLDER = Path('/content/drive/MyDrive/algoverse-jailjudge-inputs')
RUN_DIRECTORY = Path('/content/drive/MyDrive/algoverse-jailjudge-outputs/repo-screen')
import subprocess
SOURCE = Path(os.environ.get('ALGOVERSE_CHECKOUT', '/content/algoverse'))
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--branch', os.environ.get('ALGOVERSE_REF', 'main'), 'https://github.com/Dharshan2004/algoverse.git', str(SOURCE)], check=True)
commit = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip()
sys.path.insert(0, str(SOURCE))
print('Repository commit', commit)
DATA = DRIVE_INPUT_FOLDER / 'jailjudge_screen.jsonl'
if not DATA.is_file():
    # Older Drive folders keep the input JSONL alongside the original source.
    from zipfile import ZipFile
    with ZipFile(DRIVE_INPUT_FOLDER / 'baseline.zip') as archive:
        DATA = Path('/content/jailjudge_screen.jsonl')
        DATA.write_bytes(archive.read('data/processed/jailjudge_screen.jsonl'))
MODELS_TO_RUN = ['llama_guard', 'shieldgemma', 'qwen3guard', 'gemma']


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--extra-index-url', 'https://download.pytorch.org/whl/cu130',
                '-r', str(SOURCE / 'requirements.txt'), '-c', str(SOURCE / 'configs/colab-versions.txt'), 'pandas'], check=True)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from refusal_cues.jailjudge_baseline import DATA_SHA256
import hashlib
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == DATA_SHA256


In [ ]:
for model in MODELS_TO_RUN:
    for part in ('baseline', 'controls'):
        command = [sys.executable, '-m', 'scripts.run_jailjudge_baseline', model, str(DATA), str(RUN_DIRECTORY / part)]
        if part == 'controls':
            command.append('--controls')
        subprocess.run(command, cwd=SOURCE, check=True)


## Coverage and paired metrics
Gemma uses both letter mappings. Qwen scores each generation under both interpretations of Controversial.

In [ ]:
import pandas as pd
from refusal_cues import judge_metrics
from refusal_cues.protocol import PREFIXES
from refusal_cues.qwen3guard_judge import ALL_RESULTS_FIELDS
from refusal_cues.refusal_screen import DEFAULT_CONFIG

records = [json.loads(line) for line in DATA.read_bytes().splitlines()]
cue_text = {**json.loads(DEFAULT_CONFIG.read_text())['cues'], 'N': PREFIXES['N']}
rows = []
for model in MODELS_TO_RUN:
    for part in ('baseline', 'controls'):
        path = RUN_DIRECTORY / part / model / 'verdicts.jsonl'
        rows.extend(dict(json.loads(line), model=model) for line in path.read_text().splitlines())
combined = RUN_DIRECTORY / 'combined'
combined.mkdir(exist_ok=True)
pd.DataFrame(rows, columns=ALL_RESULTS_FIELDS).to_csv(combined / 'all_results.csv', index=False)
report = judge_metrics.write_report(rows, records, RUN_DIRECTORY / 'analysis', 'JailJudge development screen', cue_text)
print(report['integrity'])
